In [1]:
import torch
from functions import *

In [4]:
model = RTFEN()
total = sum(p.numel() for p in model.parameters())
print(f"模型可学习的参数量为 {total} ")

模型可学习的参数量为 13028804 


In [ ]:
# 导入模拟数据
import h5py
file = h5py.File('radar_dataset_30_50.h5', "r")
test_data = file['rdm_data'][:]                     # type: ignore
mask = file['mask_data'][:]                     # type: ignore
file.close()

np.uint64(28001)

In [ ]:
import os
import h5py
import numpy as np
import torch
import math
from scipy.ndimage import label, center_of_mass
from torch.utils.data import Dataset, DataLoader

# ====================== 导入模型文件 ======================
from functions import RTFEN, TemporalTverskyLoss

# ====================== 测试配置 ======================
TEST_DATA_PATH = 'radar_dataset_30_50.h5'  # 新测试集路径
MODEL_PATH      = 'saved_models/best_RTFEN_model.pth'
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE      = 4
THRESH          = 0.5

# ====================== 数据加载 ======================
class RadarH5Dataset(Dataset):
    def __init__(self, rdm_tensor, mask_tensor):
        self.rdm_tensor = rdm_tensor
        self.mask_tensor = mask_tensor

    def __len__(self):
        return self.rdm_tensor.shape[0]

    def __getitem__(self, idx):
        x = self.rdm_tensor[idx]
        y = self.mask_tensor[idx]

        x = torch.clamp(x, 0.0, 40.0) / 40.0
        x_tensor = x.float().unsqueeze(0)
        y_tensor = y.float().unsqueeze(1)

        return x_tensor, y_tensor

# ====================== CCA 质心评估 ======================
def evaluate_frame_with_cca(pred_mask_numpy, true_mask_numpy, distance_threshold=1.5):
    pred_labeled, pred_num_features = label(pred_mask_numpy, structure=np.ones((3,3)))
    true_labeled, true_num_features = label(true_mask_numpy, structure=np.ones((3,3)))
    
    pred_centroids = center_of_mass(pred_mask_numpy, pred_labeled, range(1, pred_num_features + 1))
    true_centroids = center_of_mass(true_mask_numpy, true_labeled, range(1, true_num_features + 1))
    
    pred_centroids = [c for c in pred_centroids if not np.isnan(c[0])]
    true_centroids = [c for c in true_centroids if not np.isnan(c[0])]

    TP, FP, FN = 0, 0, 0
    if len(true_centroids) == 0:
        return 0, len(pred_centroids), 0
    if len(pred_centroids) == 0:
        return 0, 0, len(true_centroids)

    matched_true_indices = set()
    for pred_c in pred_centroids:
        min_dist = float('inf')
        best_true_idx = -1
        for i, true_c in enumerate(true_centroids):
            dist = math.sqrt((pred_c[0] - true_c[0])**2 + (pred_c[1] - true_c[1])**2)
            if dist < min_dist:
                min_dist = dist
                best_true_idx = i
        if min_dist <= distance_threshold and best_true_idx not in matched_true_indices:
            TP += 1
            matched_true_indices.add(best_true_idx)
        else:
            FP += 1
    FN = len(true_centroids) - len(matched_true_indices)
    return TP, FP, FN

# ====================== 模型评估 ======================
def evaluate(model, dataloader, device, thresh=0.5):
    model.eval()
    total_tp, total_fp, total_fn = 0, 0, 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            preds_bin = (outputs > thresh).float().cpu().numpy()
            true_masks = targets.cpu().numpy()

            for b in range(preds_bin.shape[0]):
                for t in range(preds_bin.shape[1]):
                    tp, fp, fn = evaluate_frame_with_cca(
                        preds_bin[b, t, 0], true_masks[b, t, 0], distance_threshold=1.5
                    )
                    total_tp += tp
                    total_fp += fp
                    total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)
    f1_score = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1_score

# ====================== 主测试流程 ======================
def main():
    print("🔹 加载测试数据...")
    with h5py.File(TEST_DATA_PATH, 'r') as f:
        rdm_test  = torch.from_numpy(f['rdm_data'][:]).float() # type: ignore
        mask_test = torch.from_numpy(f['mask_data'][:]).float() # type: ignore

    test_dataset = RadarH5Dataset(rdm_test, mask_test)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    print("🔹 加载模型...")
    model = RTFEN(in_channels=1, L=32, T=50, original_size=(128, 128)).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    print("🔹 开始测试...")
    p, r, f1 = evaluate(model, test_loader, DEVICE, THRESH)

    print("\n" + "="*50)
    print("🎉 测试完成 🎉")
    print(f"Precision: {p*100:.2f}%")
    print(f"Recall:    {r*100:.2f}%")
    print(f"F1-Score:  {f1*100:.2f}%")
    print("="*50)

if __name__ == '__main__':
    main()

🔹 加载测试数据...
🔹 加载模型...
🔹 开始测试...

🎉 测试完成 🎉
Precision: 98.36%
Recall:    93.54%
F1-Score:  95.89%
